# Phase 3: Model Architecture
**Backbone:** EfficientNet-B4 (ImageNet pretrained)  
**Head:** BatchNorm → Dense(512) → Dropout → Dense(256) → Dropout → Dense(9)  
**Strategy:** 3-stage progressive fine-tuning

In [1]:
import json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_DIR    = Path(r'c:\graduation project')
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models' / 'checkpoints'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'
FIGURES_DIR = BASE_DIR / 'results' / 'figures'

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('phase3')

# Load config from Phase 2
with open(DATA_DIR / 'preprocessing_config.json') as f:
    cfg = json.load(f)

CLASSES      = cfg['classes']
NUM_CLASSES  = cfg['num_classes']
NORM_MEAN    = cfg['norm_mean']
NORM_STD     = cfg['norm_std']
IMG_SIZE     = cfg['img_size']
BATCH_SIZE   = cfg['batch_size']
weights_arr  = [cfg['class_weights'][str(i)] for i in range(NUM_CLASSES)]
CLASS_WEIGHTS = torch.tensor(weights_arr, dtype=torch.float32).to(DEVICE)
MALIGNANT    = {'melanoma', 'basal cell carcinoma', 'squamous cell carcinoma'}

print(f"Device      : {DEVICE}")
print(f"Classes     : {NUM_CLASSES}")
print(f"Image size  : {IMG_SIZE}x{IMG_SIZE}")

Device      : cuda
Classes     : 8
Image size  : 224x224


## 1 · Rebuild DataLoaders (from Phase 2 splits)

In [2]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = row.get('preprocessed_path') or row['image_path']
        image = Image.open(path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, int(row['label_int'])


train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(degrees=15),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.90, 1.0)),
    T.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])
val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

train_loader = DataLoader(SkinLesionDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(SkinLesionDataset(val_df,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(SkinLesionDataset(test_df,  val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

log.info(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

12:19:01  Train: 7,465  Val: 1,600  Test: 1,715


## 2 · Model Architecture

```
EfficientNet-B4 backbone (pretrained ImageNet)
         ↓
   AdaptiveAvgPool → Flatten   [1792-dim]
         ↓
   BatchNorm1d(1792)
         ↓
   Linear(1792 → 512) + ReLU + Dropout(0.4)
         ↓
   Linear(512 → 256) + ReLU + Dropout(0.3)
         ↓
   Linear(256 → 9)        ← 9-class output
```

In [3]:
class SkinCancerModel(nn.Module):
    """
    EfficientNet-B4 backbone + custom 3-layer classification head.
    Supports 3-stage progressive fine-tuning via set_stage().
    """
    IN_FEATURES = 1792   # EfficientNet-B4 feature dim after global avg pool

    def __init__(self, num_classes: int = 9):
        super().__init__()
        base = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

        self.features = base.features   # convolutional backbone
        self.avgpool  = base.avgpool    # AdaptiveAvgPool2d(1)

        # Custom classification head
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.IN_FEATURES),
            nn.Linear(self.IN_FEATURES, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

        # Start with backbone fully frozen (Stage 1)
        self.set_stage(1)

    # ── Fine-tuning stages ────────────────────────────────────────────────────
    def set_stage(self, stage: int):
        """
        Stage 1 — head only (backbone frozen)
        Stage 2 — head + last 4 backbone blocks unfrozen
        Stage 3 — full model unfrozen
        """
        # Freeze entire backbone first
        for p in self.features.parameters():
            p.requires_grad = False

        if stage == 2:
            # Unfreeze features[5..8] — last 4 blocks of EfficientNet-B4
            for block in list(self.features.children())[5:]:
                for p in block.parameters():
                    p.requires_grad = True

        elif stage == 3:
            # Unfreeze everything
            for p in self.features.parameters():
                p.requires_grad = True

        # Head is always trainable
        for p in self.classifier.parameters():
            p.requires_grad = True

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        log.info(f"Stage {stage} — trainable: {trainable:,} / {total:,} "
                 f"({100*trainable/total:.1f}%)")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x   # raw logits — loss function applies softmax internally


print("SkinCancerModel defined ✓")

SkinCancerModel defined ✓


## 3 · Build Stage 1 Model & Verify

In [4]:
model = SkinCancerModel(num_classes=NUM_CLASSES).to(DEVICE)

# Parameter summary
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print("\nParameter count:")
print(f"  Total      : {total:>12,}")
print(f"  Trainable  : {trainable:>12,}  (head only — Stage 1)")
print(f"  Frozen     : {frozen:>12,}  (EfficientNet-B4 backbone)")

# Forward pass test
dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)

print(f"\nForward pass check:")
print(f"  Input  : {list(dummy.shape)}")
print(f"  Output : {list(out.shape)}   ← (batch, 9 classes)")
assert out.shape == (4, NUM_CLASSES), "Output shape mismatch!"

# Softmax probabilities sum to 1 per sample
probs = torch.softmax(out, dim=1)
assert torch.allclose(probs.sum(dim=1), torch.ones(4).to(DEVICE), atol=1e-5)
print(f"  Softmax sums to 1.0 per sample ✓")

12:19:01  Stage 1 — trainable: 1,054,984 / 18,603,600 (5.7%)



Parameter count:
  Total      :   18,603,600
  Trainable  :    1,054,984  (head only — Stage 1)
  Frozen     :   17,548,616  (EfficientNet-B4 backbone)

Forward pass check:
  Input  : [4, 3, 224, 224]
  Output : [4, 8]   ← (batch, 9 classes)
  Softmax sums to 1.0 per sample ✓


## 4 · 3-Stage Fine-Tuning Strategy

In [5]:
# Preview trainable parameter counts for all 3 stages
print("Fine-tuning stage overview:")
print(f"{'Stage':<8} {'Trainable params':>18} {'% of total':>12} {'What trains'}")
print("-" * 68)

STAGE_CFG = {
    1: {'epochs': 10,  'lr': 1e-3,  'desc': 'Head only'},
    2: {'epochs': 20,  'lr': 1e-4,  'desc': 'Head + last 4 backbone blocks'},
    3: {'epochs': 10,  'lr': 1e-5,  'desc': 'Full model'},
}

for stage, scfg in STAGE_CFG.items():
    model.set_stage(stage)
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {stage:<6} {tr:>18,} {100*tr/total:>11.1f}%   "
          f"{scfg['desc']}  (lr={scfg['lr']}, {scfg['epochs']} epochs)")

# Reset to Stage 1 for training
model.set_stage(1)

12:19:01  Stage 1 — trainable: 1,054,984 / 18,603,600 (5.7%)
12:19:01  Stage 2 — trainable: 17,274,340 / 18,603,600 (92.9%)
12:19:01  Stage 3 — trainable: 18,603,600 / 18,603,600 (100.0%)
12:19:01  Stage 1 — trainable: 1,054,984 / 18,603,600 (5.7%)


Fine-tuning stage overview:
Stage      Trainable params   % of total What trains
--------------------------------------------------------------------
  1               1,054,984         5.7%   Head only  (lr=0.001, 10 epochs)
  2              17,274,340        92.9%   Head + last 4 backbone blocks  (lr=0.0001, 20 epochs)
  3              18,603,600       100.0%   Full model  (lr=1e-05, 10 epochs)


## 5 · Loss Function

In [6]:
# CrossEntropy with label smoothing + class weights
# label_smoothing=0.1 prevents overconfident predictions — critical in medical AI
criterion = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    label_smoothing=0.1
)

# Test loss on a dummy batch
dummy_logits = torch.randn(8, NUM_CLASSES).to(DEVICE)
dummy_labels = torch.randint(0, NUM_CLASSES, (8,)).to(DEVICE)
loss_val = criterion(dummy_logits, dummy_labels)
print(f"Loss function  : CrossEntropyLoss (label_smoothing=0.1, weighted)")
print(f"Test loss value: {loss_val.item():.4f}  (random inputs — sanity check)")
print(f"Class weights on {DEVICE}:")
for i, (cls, w) in enumerate(zip(CLASSES, CLASS_WEIGHTS.cpu())):
    tag = ' ← MALIGNANT' if cls in MALIGNANT else ''
    print(f"  {i}  {cls:<35} {w:.4f}{tag}")

Loss function  : CrossEntropyLoss (label_smoothing=0.1, weighted)
Test loss value: 2.7518  (random inputs — sanity check)
Class weights on cuda:
  0  actinic keratosis                   11.1086
  1  basal cell carcinoma                2.6737 ← MALIGNANT
  2  dermatofibroma                      12.6098
  3  melanoma                            0.8592 ← MALIGNANT
  4  nevus                               0.1909
  5  pigmented benign keratosis          1.2294
  6  squamous cell carcinoma             7.3474 ← MALIGNANT
  7  vascular lesion                     9.6198


## 6 · Optimiser Strategy

In [7]:
def build_optimizer(model: nn.Module, stage: int) -> torch.optim.Optimizer:
    """
    Separate parameter groups so backbone LR can be lower than head LR.
    Stage 1: only head trained
    Stage 2: head at lr, backbone (unfrozen part) at lr/10
    Stage 3: head at lr, full backbone at lr/10
    """
    base_lr = STAGE_CFG[stage]['lr']

    if stage == 1:
        return torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=base_lr, weight_decay=1e-4
        )
    else:
        head_params     = list(model.classifier.parameters())
        backbone_params = [p for p in model.features.parameters() if p.requires_grad]
        return torch.optim.Adam([
            {'params': head_params,     'lr': base_lr},
            {'params': backbone_params, 'lr': base_lr / 10},
        ], weight_decay=1e-4)


def build_scheduler(optimizer: torch.optim.Optimizer,
                    stage: int) -> torch.optim.lr_scheduler._LRScheduler:
    """CosineAnnealingLR for stages 2 and 3; StepLR warmup-like for stage 1."""
    epochs = STAGE_CFG[stage]['epochs']
    if stage == 1:
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=STAGE_CFG[1]['lr'],
            steps_per_epoch=len(train_loader), epochs=epochs,
            pct_start=0.2
        )
    else:
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-7
        )


# Quick test
optimizer = build_optimizer(model, stage=1)
scheduler = build_scheduler(optimizer, stage=1)
print(f"Stage 1 optimiser: {type(optimizer).__name__}  lr={STAGE_CFG[1]['lr']}")
print(f"Stage 1 scheduler: {type(scheduler).__name__}")

Stage 1 optimiser: Adam  lr=0.001
Stage 1 scheduler: OneCycleLR


## 7 · Backbone Architecture Visualisation

In [8]:
# Show which backbone blocks exist and which get unfrozen per stage
print("EfficientNet-B4 backbone blocks:")
print(f"  {'Block':>6}  {'Name':<40}  {'Stage 1':>8}  {'Stage 2':>8}  {'Stage 3':>8}")
print("  " + "-" * 74)

for i, (name, module) in enumerate(model.features.named_children()):
    n_params = sum(p.numel() for p in module.parameters())
    s1 = 'frozen'
    s2 = 'TRAIN' if i >= 5 else 'frozen'
    s3 = 'TRAIN'
    print(f"  {i:>6}  {str(type(module).__name__):<40}  {s1:>8}  {s2:>8}  {s3:>8}  ({n_params:,} params)")

print("\n  Head (classifier): TRAIN in all stages")

EfficientNet-B4 backbone blocks:
   Block  Name                                       Stage 1   Stage 2   Stage 3
  --------------------------------------------------------------------------
       0  Conv2dNormActivation                        frozen    frozen     TRAIN  (1,392 params)
       1  Sequential                                  frozen    frozen     TRAIN  (4,146 params)
       2  Sequential                                  frozen    frozen     TRAIN  (66,238 params)
       3  Sequential                                  frozen    frozen     TRAIN  (197,586 params)
       4  Sequential                                  frozen    frozen     TRAIN  (1,059,898 params)
       5  Sequential                                  frozen     TRAIN     TRAIN  (2,306,724 params)
       6  Sequential                                  frozen     TRAIN     TRAIN  (8,636,228 params)
       7  Sequential                                  frozen     TRAIN     TRAIN  (4,470,004 params)
       8  Conv

## 8 · Save Model Architecture Info

In [9]:
arch_info = {
    'backbone'        : 'EfficientNet-B4',
    'pretrained'      : 'ImageNet (EfficientNet_B4_Weights.IMAGENET1K_V1)',
    'input_size'      : [3, IMG_SIZE, IMG_SIZE],
    'num_classes'     : NUM_CLASSES,
    'in_features'     : SkinCancerModel.IN_FEATURES,
    'head'            : [
        'BatchNorm1d(1792)',
        'Linear(1792→512) + ReLU + Dropout(0.4)',
        'Linear(512→256) + ReLU + Dropout(0.3)',
        'Linear(256→9)',
    ],
    'total_params'    : total,
    'stages'          : STAGE_CFG,
    'loss'            : 'CrossEntropyLoss(label_smoothing=0.1, weighted)',
}

arch_path = METRICS_DIR / 'model_architecture.json'
with open(arch_path, 'w') as f:
    json.dump(arch_info, f, indent=2, default=str)
log.info(f"Saved → {arch_path}")

# Save model weights (Stage 1 initialised)
init_path = MODELS_DIR / 'model_init.pth'
torch.save(model.state_dict(), init_path)
log.info(f"Saved initial weights → {init_path}")

print("\nModel architecture info saved ✓")

12:19:01  Saved → c:\graduation project\results\metrics\model_architecture.json
12:19:01  Saved initial weights → c:\graduation project\models\checkpoints\model_init.pth



Model architecture info saved ✓


## Phase 3 Complete ✓

| Item | Detail |
|------|--------|
| Backbone | EfficientNet-B4, ImageNet pretrained |
| Head | BN → 512 → 256 → 9 (logits) |
| Stage 1 | Head only — 10 epochs, lr=1e-3 |
| Stage 2 | Head + last 4 blocks — 20 epochs, lr=1e-4 |
| Stage 3 | Full model — 10 epochs, lr=1e-5 |
| Loss | CrossEntropy + label_smoothing=0.1 + class weights |
| Outputs | `models/checkpoints/model_init.pth`, `results/metrics/model_architecture.json` |

**Next → Phase 4: Training** — full 3-stage training loop with checkpoints, early stopping, and melanoma sensitivity tracking